**Model Comparison**

In [22]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import TransformedTargetRegressor

import joblib

Load Train/Test CSV's

In [23]:
output_dir = Path("outputs")

train_df = pd.read_csv(output_dir / "train_202605_split.csv", low_memory=False)
test_df = pd.read_csv(output_dir / "test_202605_split.csv", low_memory=False)

with open(output_dir / "preprocessing_metadata.json", "r") as f:
    metadata = json.load(f)

target_col = metadata["target_col"]

numeric_features = metadata["numeric_features"] + metadata["missing_flag_cols"]
categorical_features = metadata["categorical_features"] + metadata["boolean_features"]

# Keep only columns that actually exist
numeric_features = [col for col in numeric_features if col in train_df.columns]
categorical_features = [col for col in categorical_features if col in train_df.columns]

feature_cols = numeric_features + categorical_features

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Number of features before encoding:", len(feature_cols))

Train shape: (386862, 41)
Test shape: (12017, 41)
Number of features before encoding: 34


In [24]:
X_train = train_df[feature_cols].copy()
y_train = train_df[target_col].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df[target_col].copy()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (386862, 34)
y_train: (386862,)
X_test: (12017, 34)
y_test: (12017,)


Build Preprocessing pipelines for both linear and tree models

In [25]:
numeric_pipeline_linear = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

numeric_pipeline_tree = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_linear = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline_linear, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ]
)

preprocessor_tree = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline_tree, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ]
)

Define models 

In [26]:
models = {
    "Linear Regression": Pipeline(steps=[
        ("preprocess", preprocessor_linear),
        ("model", LinearRegression())
    ]),

    "Decision Tree": Pipeline(steps=[
        ("preprocess", preprocessor_tree),
        ("model", DecisionTreeRegressor(
            max_depth=20,
            min_samples_leaf=25,
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline(steps=[
        ("preprocess", preprocessor_tree),
        ("model", RandomForestRegressor(
            n_estimators=50,
            max_depth=20,
            min_samples_leaf=10,
            random_state=42,
            n_jobs=-1
        ))
    ])
}

Train and Evaluate each model

In [27]:
def evaluate_model(model_name, model_pipeline, X_train, y_train, X_test, y_test):
    start_time = time.time()

    model_pipeline.fit(X_train, y_train)
    fit_seconds = time.time() - start_time

    y_pred = model_pipeline.predict(X_test)

    residuals = y_test.to_numpy() - y_pred

    r2 = r2_score(y_test, y_pred)
    correlation_r = np.corrcoef(y_test, y_pred)[0, 1]
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    metrics = {
        "model": model_name,
        "train_rows": len(X_train),
        "test_rows": len(X_test),
        "test_r2": r2,
        "correlation_r": correlation_r,
        "mae": mae,
        "rmse": rmse,
        "fit_seconds": fit_seconds
    }

    return metrics, y_pred, residuals, model_pipeline

In [28]:
results = []
predictions = {}
residual_dict = {}
fitted_models = {}

for model_name, model_pipeline in models.items():
    print(f"Training {model_name}...")

    metrics, y_pred, residuals, fitted_model = evaluate_model(
        model_name,
        model_pipeline,
        X_train,
        y_train,
        X_test,
        y_test
    )

    results.append(metrics)
    predictions[model_name] = y_pred
    residual_dict[model_name] = residuals
    fitted_models[model_name] = fitted_model

    print(f"{model_name} test R²: {metrics['test_r2']:.4f}")
    print()

results_df = pd.DataFrame(results)

baseline_r2 = results_df.loc[
    results_df["model"] == "Linear Regression",
    "test_r2"
].iloc[0]

results_df["r2_vs_baseline"] = results_df["test_r2"] - baseline_r2

results_df = results_df.sort_values("test_r2", ascending=False).reset_index(drop=True)

results_df

Training Linear Regression...
Linear Regression test R²: 0.4244

Training Decision Tree...
Decision Tree test R²: -0.0768

Training Random Forest...
Random Forest test R²: 0.1836



,model,train_rows,test_rows,test_r2,correlation_r,mae,rmse,fit_seconds,r2_vs_baseline
0,Linear Regression,386862,12017,0.424431,0.651906,372465.599406,1.273387e+06,20.843425,0.000000
1,Random Forest,386862,12017,0.183599,0.557412,286907.979087,1.516573e+06,162.416454,-0.240831
2,Decision Tree,386862,12017,-0.076755,0.496756,338447.914057,1.741687e+06,14.160399,-0.501186


Linear Regression performed the best with an r2 of 0.4244, but it is still not ideal. 

Random Forest had a worse r2 of 0.1836 but had a lower MAE than the other two models, meaning it is making accurate predictions on typical homes, but making large errors on house prices that are outliers

Decision Tree performed the worst with an r2 of -0.0768, which means it's predicting worse than a model that simply predicts the mean close price for every house. This is most likely because the distribution of ClosePrice is skewed and the location features have high-cardinality.

Next Steps: tune random forest hyperparameters, test a log-transformed target, and add stronger features for geographic features. 

In [29]:
for model_name, model in models.items():
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_r2 = r2_score(y_train, train_pred)
    test_r2 = r2_score(y_test, test_pred)

    print(model_name)
    print("Train R²:", train_r2)
    print("Test R²:", test_r2)
    print()

Linear Regression
Train R²: 0.13553188246411474
Test R²: 0.42443053624612515

Decision Tree
Train R²: 0.09495135998933579
Test R²: -0.07675530898707983

Random Forest
Train R²: 0.15643325865111068
Test R²: 0.1835991083664983



The Linear Regression and Random Forest models train r2 are lower than their test r2. This might possibly be due to the train data being far larger and more diverse compared to the much smaller test set with less variability. Both models are underfitting.

Trying for a better Random Forest

In [30]:
random_forest_tuned = Pipeline(steps=[
    ("preprocess", preprocessor_tree),
    ("model", RandomForestRegressor(
        n_estimators=100,
        max_depth=None,
        min_samples_leaf=5,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    ))
])

In [31]:
random_forest_tuned.fit(X_train, y_train)

# Predict
train_pred = random_forest_tuned.predict(X_train)
test_pred = random_forest_tuned.predict(X_test)

# Metrics
train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)
correlation_r = np.corrcoef(y_test, test_pred)[0, 1]
mae = mean_absolute_error(y_test, test_pred)
rmse = np.sqrt(mean_squared_error(y_test, test_pred))

# Output metrics
print("Random Forest Tuned Performance")
print("Train R²:", train_r2)
print("Test R²:", test_r2)
print("Correlation r:", correlation_r)
print("MAE:", mae)
print("RMSE:", rmse)

Random Forest Tuned Performance
Train R²: 0.10529964987471341
Test R²: 0.43006490518251617
Correlation r: 0.6753803677825311
MAE: 338907.3602861793
RMSE: 1267138.8768224295


The original Random Forest Model had min_samples_leaf of 10, which means each final leaf must contain at least 10 samples. This was smoothing away important local price differences.

**Log-transforming the target feature**

Making sure all ClosePrice values are positive before applying log transformation

In [32]:
print("Minimum y_train:", y_train.min())
print("Minimum y_test:", y_test.min())

assert (y_train > 0).all(), "y_train contains zero or negative ClosePrice values."
assert (y_test > 0).all(), "y_test contains zero or negative ClosePrice values."

Minimum y_train: 1.15
Minimum y_test: 11900.0


In [33]:
log_models = {
    "Linear Regression": Pipeline(steps=[
        ("preprocess", preprocessor_linear),
        ("model", TransformedTargetRegressor(
            regressor=LinearRegression(),
            func=np.log1p,
            inverse_func=np.expm1
        ))
    ]),

    "Decision Tree": Pipeline(steps=[
        ("preprocess", preprocessor_tree),
        ("model", TransformedTargetRegressor(
            regressor=DecisionTreeRegressor(
                max_depth=20,
                min_samples_leaf=25,
                random_state=42
            ),
            func=np.log1p,
            inverse_func=np.expm1
        ))
    ]),

    "Random Forest Tuned": Pipeline(steps=[
        ("preprocess", preprocessor_tree),
        ("model", TransformedTargetRegressor(
            regressor=RandomForestRegressor(
                n_estimators=100,
                max_depth=None,
                min_samples_leaf=5,
                max_features="sqrt",
                random_state=42,
                n_jobs=-1
            ),
            func=np.log1p,
            inverse_func=np.expm1
        ))
    ])
}

Note: TransformedTargetRegressor trains the model on the log of ClosePrice, but converts it back into $ amount

In [34]:
log_results = []
log_predictions = {}
log_residual_dict = {}
log_fitted_models = {}

for model_name, model_pipeline in log_models.items():
    print(f"Training {model_name} (log-transformed target)...")

    metrics, y_pred, residuals, fitted_model = evaluate_model(
        model_name,
        model_pipeline,
        X_train,
        y_train,
        X_test,
        y_test
    )

    log_results.append(metrics)
    log_predictions[model_name] = y_pred
    log_residual_dict[model_name] = residuals
    log_fitted_models[model_name] = fitted_model

    print(f"{model_name} (log target) test R\u00b2: {metrics['test_r2']:.4f}")
    print()

Training Linear Regression (log-transformed target)...
Linear Regression (log target) test R²: 0.4671

Training Decision Tree (log-transformed target)...
Decision Tree (log target) test R²: 0.5092

Training Random Forest Tuned (log-transformed target)...
Random Forest Tuned (log target) test R²: 0.3400



In [35]:
def compute_metrics(name, target_type, fitted_pipeline, X_train, y_train, X_test, y_test, fit_seconds=None):
    train_pred = fitted_pipeline.predict(X_train)
    test_pred = fitted_pipeline.predict(X_test)

    return {
        "model": name,
        "target": target_type,
        "train_r2": r2_score(y_train, train_pred),
        "test_r2": r2_score(y_test, test_pred),
        "correlation_r": np.corrcoef(y_test, test_pred)[0, 1],
        "mae": mean_absolute_error(y_test, test_pred),
        "rmse": np.sqrt(mean_squared_error(y_test, test_pred)),
        "fit_seconds": fit_seconds
    }

# Raw-target models, matched to the same hyperparameters used in log_models
raw_counterparts = {
    "Linear Regression": (models["Linear Regression"], None),
    "Decision Tree": (models["Decision Tree"], None),
    "Random Forest Tuned": (random_forest_tuned, None),
}

comparison_rows = []

for model_name, (fitted_pipeline, fit_seconds) in raw_counterparts.items():
    comparison_rows.append(
        compute_metrics(model_name, "Raw", fitted_pipeline, X_train, y_train, X_test, y_test, fit_seconds)
    )

for model_name, fitted_pipeline in log_fitted_models.items():
    fit_seconds = next(r["fit_seconds"] for r in log_results if r["model"] == model_name)
    comparison_rows.append(
        compute_metrics(model_name, "Log", fitted_pipeline, X_train, y_train, X_test, y_test, fit_seconds)
    )

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

,model,target,train_r2,test_r2,correlation_r,mae,rmse,fit_seconds
0,Linear Regression,Raw,1.355319e-01,0.424431,0.651906,372465.599406,1.273387e+06,NaN
1,Decision Tree,Raw,9.495136e-02,-0.076755,0.496756,338447.914057,1.741687e+06,NaN
2,Random Forest Tuned,Raw,1.052996e-01,0.430065,0.675380,338907.360286,1.267139e+06,NaN
3,Linear Regression,Log,-8.190321e+09,0.467090,0.686079,260193.770909,1.225289e+06,28.336294
4,Decision Tree,Log,5.697825e-02,0.509150,0.717207,245232.904098,1.175942e+06,18.305463
5,Random Forest Tuned,Log,3.425041e-02,0.340033,0.666979,336394.966832,1.363555e+06,21.264156


In [ ]:
pivot_df = comparison_df.pivot(index="model", columns="target",
                                values=["test_r2", "mae", "rmse", "fit_seconds"])

# Flatten the column MultiIndex into readable names
pivot_df.columns = [f"{metric}_{target.lower()}" for metric, target in pivot_df.columns]

summary_table = pd.DataFrame({
    "Raw R\u00b2": pivot_df["test_r2_raw"],
    "Log R\u00b2": pivot_df["test_r2_log"],
    "R\u00b2 change": pivot_df["test_r2_log"] - pivot_df["test_r2_raw"],
    "Raw MAE ($)": pivot_df["mae_raw"],
    "Log MAE ($)": pivot_df["mae_log"],
    "MAE change ($)": pivot_df["mae_log"] - pivot_df["mae_raw"],
    "Raw RMSE ($)": pivot_df["rmse_raw"],
    "Log RMSE ($)": pivot_df["rmse_log"],
    "RMSE change ($)": pivot_df["rmse_log"] - pivot_df["rmse_raw"],
})

summary_table = summary_table.reindex(["Linear Regression", "Decision Tree", "Random Forest Tuned"])

summary_table.style.format({
    "Raw R\u00b2": "{:.4f}", "Log R\u00b2": "{:.4f}", "R\u00b2 change": "{:+.4f}",
    "Raw MAE ($)": "{:,.0f}", "Log MAE ($)": "{:,.0f}", "MAE change ($)": "{:+,.0f}",
    "Raw RMSE ($)": "{:,.0f}", "Log RMSE ($)": "{:,.0f}", "RMSE change ($)": "{:+,.0f}",
}).background_gradient(subset=["R\u00b2 change"], cmap="RdYlGn")

,Raw R²,Log R²,R² change,Raw MAE ($),Log MAE ($),MAE change ($),Raw RMSE ($),Log RMSE ($),RMSE change ($)
model,,,,,,,,,
Linear Regression,0.4244,0.4671,+0.0427,"372,466","260,194","-112,272","1,273,387","1,225,289","-48,098"
Decision Tree,-0.0768,0.5092,+0.5859,"338,448","245,233","-93,215","1,741,687","1,175,942","-565,745"
Random Forest Tuned,0.4301,0.3400,-0.0900,"338,907","336,395","-2,512","1,267,139","1,363,555","+96,417"


**Model Comparison Reflection**

**Initial performance:** 
-The first pass across three baseline models exposed a distribution problem more than a modeling problem. 

-Linear Regression led with R² = 0.4244

-Random Forest (default config: min_samples_leaf=10) badly underperformed it at R² = 0.1836

-Decision Tree scored negative R² (-0.0768)

These performance metrics indicate  that ClosePrice is heavily right-skewed: a small number of high-value homes dominate the variance

**Changes and their effects: **

-Random Forest Tuning: n_estimators 50->100, min_samples_leaf 10->5, max_features="sqrt", unbounded depth led to an inrease of +0.58 in r2

**-Log Transforming the Target:**

-Decision Tree: R2 went up by +0.58 and RMSE dropped by 560K. Log transforming the target pulled the high-price outliers close enough to the bulk of the distribution that a single tree could finally carve out sensible splits instead of being dominated by a handful of extreme values.

-Linear Regression: R2 went up by +0.04 and RMSE went down by 48K. home prices are closer to log-normal than normal, so modeling log-price with a linear model is a better-specified functional form than modeling raw price linearly.

-Random Forest Tuned: R2 went down by -0.09 and RMSE increased by 96K. Random Forest already gets some robustness to skew "for free" through averaging across trees, so log-transforming didn't fix a problem it didn't fully have. Instead, the forest ended up optimizing splits for log-scale variance reduction, which doesn't necessarily minimize dollar-scale error, particularly for the high-end homes